In [7]:
# ============================================================
# Stage 1 | Cell 1 — Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount("/content/drive")
print("Google Drive mounted successfully.")

Mounted at /content/drive
Google Drive mounted successfully.


In [ ]:
#now we will veryofy all paths valid

In [8]:
# ============================================================
# Stage 1 | Cell 2 — Define Project Paths
# ============================================================
import os

# Root of your project inside Google Drive
PROJECT_ROOT = "/content/drive/MyDrive/BioBERT_Project"

# Sub-folders
DATA_DIR        = os.path.join(PROJECT_ROOT, "data")
CHECKPOINT_DIR  = os.path.join(PROJECT_ROOT, "checkpoints")
LOG_DIR         = os.path.join(PROJECT_ROOT, "logs")
OUTPUT_DIR      = os.path.join(PROJECT_ROOT, "outputs")

# The zip file
ZIP_PATH        = os.path.join(PROJECT_ROOT, "split_dataset.zip")

# Confirm all expected paths exist
print("Checking project structure...\n")
paths_to_check = {
    "Project root":   PROJECT_ROOT,
    "data/":          DATA_DIR,
    "checkpoints/":   CHECKPOINT_DIR,
    "logs/":          LOG_DIR,
    "outputs/":       OUTPUT_DIR,
    "ZIP file":       ZIP_PATH,
}

all_ok = True
for name, path in paths_to_check.items():
    exists = os.path.exists(path)
    status = "OK" if exists else "MISSING"
    print(f"  {status}   {name}: {path}")
    if not exists:
        all_ok = False

print()
if all_ok:
    print("All paths verified. Ready to proceed.")
else:
    print("Some paths are missing. Fix them before continuing.")

Checking project structure...

  OK   Project root: /content/drive/MyDrive/BioBERT_Project
  OK   data/: /content/drive/MyDrive/BioBERT_Project/data
  OK   checkpoints/: /content/drive/MyDrive/BioBERT_Project/checkpoints
  OK   logs/: /content/drive/MyDrive/BioBERT_Project/logs
  OK   outputs/: /content/drive/MyDrive/BioBERT_Project/outputs
  OK   ZIP file: /content/drive/MyDrive/BioBERT_Project/split_dataset.zip

All paths verified. Ready to proceed.


In [ ]:
# now install dependecnies

In [ ]:
print("Installing dependencies... (this takes ~1 minute)")

!pip install --quiet \
    transformers==4.40.0 \
    torch \
    scikit-learn \
    pandas \
    numpy

print("Installation complete.")

Installing dependencies... (this takes ~1 minute)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 100.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 126.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.6.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
Installation complete.


In [ ]:
# verify GPU

In [9]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")

if device.type == "cuda":
    print(f"GPU Name:      {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print()
    print("WARNING: No GPU detected.")
    print("Go to Runtime > Change runtime type > Hardware accelerator > GPU")
    print("Then restart and run all cells again.")

Device: cuda
GPU Name:      Tesla T4
GPU Memory:    15.6 GB


In [ ]:
#STage 2 unloading the data

In [ ]:
# UnZip the dataset folder split_dataset which is now in zip form and unzip files into /data folder in drive

In [10]:
import zipfile
import os

print(f"Extracting: {ZIP_PATH}")
print(f"Into:       {DATA_DIR}\n")

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    # Show what's inside the zip before extracting
    contents = zf.namelist()
    print("Files inside zip:")
    for f in contents:
        print(f"  {f}")
    print()

    # Extract everything into data/
    zf.extractall(DATA_DIR)

print("Extraction complete.")

Extracting: /content/drive/MyDrive/BioBERT_Project/split_dataset.zip
Into:       /content/drive/MyDrive/BioBERT_Project/data

Files inside zip:
  split_dataset/
  split_dataset/test.csv
  split_dataset/train.csv
  split_dataset/validation.csv

Extraction complete.


In [ ]:
#load the csv and verify their structure

In [11]:
import pandas as pd


TRAIN_PATH = os.path.join(DATA_DIR, "split_dataset/train.csv")
VAL_PATH   = os.path.join(DATA_DIR, "split_dataset/validation.csv")
TEST_PATH  = os.path.join(DATA_DIR, "split_dataset/test.csv")

# Load
train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("=" * 50)
print("SHAPES")
print("=" * 50)
print(f"  Train:      {train_df.shape}")
print(f"  Validation: {val_df.shape}")
print(f"  Test:       {test_df.shape}")

print("\n" + "=" * 50)
print("COLUMN NAMES")
print("=" * 50)
for name, df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"  {name}: {df.columns.tolist()}")

print("\n" + "=" * 50)
print("FIRST TWO ROWS (train)")
print("=" * 50)
print(train_df.head(2).to_string())

SHAPES
  Train:      (2636, 3)
  Validation: (565, 3)
  Test:       (565, 3)

COLUMN NAMES
  Train: ['text', 'disease', 'label']
  Val: ['text', 'disease', 'label']
  Test: ['text', 'disease', 'label']

FIRST TWO ROWS (train)
                                                                                            text    disease  label
0  The patient presents with shortness of breath, wheezing, weakness, fever, chills, and coryza.  pneumonia      4
1                    The patient presents with nasal congestion, vomiting, wheezing, and chills.  pneumonia      4


In [ ]:
#Check Class distributions per disease label becasue we have to handle the minority classed next

In [12]:
TEXT_COL    = "text"
DISEASE_COL = "disease"
LABEL_COL   = "label"

print("=" * 50)
print("CLASS DISTRIBUTION — TRAIN")
print("=" * 50)
dist = train_df.groupby([LABEL_COL, DISEASE_COL]).size().reset_index(name="count")
dist = dist.sort_values(LABEL_COL)
for _, row in dist.iterrows():
    bar = "█" * (row["count"] // 30)
    print(f"  [{int(row[LABEL_COL])}] {row[DISEASE_COL]:<25} {row['count']:>4}  {bar}")

print("\n" + "=" * 50)
print("LABEL CONSISTENCY CHECK")
print("=" * 50)

# Build label→disease map from train, then confirm val and test use the same mapping
train_map = dict(zip(train_df[LABEL_COL], train_df[DISEASE_COL]))
train_map = {k: train_map[k] for k in sorted(train_map)}

val_map  = dict(zip(val_df[LABEL_COL],  val_df[DISEASE_COL]))
test_map = dict(zip(test_df[LABEL_COL], test_df[DISEASE_COL]))

print("\n  Label → Disease mapping (from train):")
for label, disease in train_map.items():
    print(f"    {label} → {disease}")

val_match  = all(val_map.get(k)  == v for k, v in train_map.items())
test_match = all(test_map.get(k) == v for k, v in train_map.items())

print(f"\n  Val  mapping matches train: {val_match}")
print(f"  Test mapping matches train: {test_match}")

print("\n" + "=" * 50)
print("MISSING VALUES CHECK")
print("=" * 50)
for name, df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    nulls = df[[TEXT_COL, LABEL_COL]].isnull().sum()
    print(f"  {name}: text nulls = {nulls[TEXT_COL]}, label nulls = {nulls[LABEL_COL]}")

print("\n" + "=" * 50)
print("LABEL DTYPE CHECK")
print("=" * 50)
print(f"  Train label dtype: {train_df[LABEL_COL].dtype}")
print(f"  Sample text: \"{train_df[TEXT_COL].iloc[0]}\"")

CLASS DISTRIBUTION — TRAIN
  [0] atelectasis                 69  ██
  [1] emphysema                   22  
  [2] hiatal hernia              634  █████████████████████
  [3] pleural effusion           425  ██████████████
  [4] pneumonia                  848  ████████████████████████████
  [5] pneumothorax               207  ██████
  [6] pulmonary congestion       348  ███████████
  [7] pulmonary fibrosis          83  ██

LABEL CONSISTENCY CHECK

  Label → Disease mapping (from train):
    0 → atelectasis
    1 → emphysema
    2 → hiatal hernia
    3 → pleural effusion
    4 → pneumonia
    5 → pneumothorax
    6 → pulmonary congestion
    7 → pulmonary fibrosis

  Val  mapping matches train: True
  Test mapping matches train: True

MISSING VALUES CHECK
  Train: text nulls = 0, label nulls = 0
  Val: text nulls = 0, label nulls = 0
  Test: text nulls = 0, label nulls = 0

LABEL DTYPE CHECK
  Train label dtype: int64
  Sample text: "The patient presents with shortness of breath, wheezing,

In [ ]:
#stage3 Tokenization and Dataset class

In [ ]:
#load tokenizer and inspect a sample

In [13]:
# ============================================================
# Stage 3 | Cell 8 — Load BioBERT Tokenizer and Inspect
# ============================================================
from transformers import BertTokenizer

MODEL_NAME = "dmis-lab/biobert-base-cased-v1.1"
MAX_LENGTH = 128

print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer loaded.\n")

# ── Inspect one real sample from your training data ──
sample_text  = train_df["text"].iloc[0]
sample_label = train_df["label"].iloc[0]

print("=" * 55)
print("SAMPLE INSPECTION")
print("=" * 55)
print(f"  Text:    {sample_text}")
print(f"  Label:   {sample_label}")
print()

encoding = tokenizer(
    sample_text,
    max_length=MAX_LENGTH,
    padding="max_length",
    truncation=True,
    return_tensors="pt"
)

input_ids      = encoding["input_ids"].squeeze(0)
attention_mask = encoding["attention_mask"].squeeze(0)

# Count real tokens vs padding tokens
real_tokens    = attention_mask.sum().item()
pad_tokens     = MAX_LENGTH - real_tokens

print(f"  input_ids shape:      {input_ids.shape}")
print(f"  attention_mask shape: {attention_mask.shape}")
print(f"  Real tokens:          {real_tokens}  (including [CLS] and [SEP])")
print(f"  Padding tokens:       {pad_tokens}")
print()

# Decode the tokens back to words so you can see what BioBERT sees
tokens = tokenizer.convert_ids_to_tokens(input_ids[:real_tokens])
print(f"  Tokens BioBERT sees: {tokens}")

Loading tokenizer: dmis-lab/biobert-base-cased-v1.1


vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

Tokenizer loaded.

SAMPLE INSPECTION
  Text:    The patient presents with shortness of breath, wheezing, weakness, fever, chills, and coryza.
  Label:   4

  input_ids shape:      torch.Size([128])
  attention_mask shape: torch.Size([128])
  Real tokens:          27  (including [CLS] and [SEP])
  Padding tokens:       101

  Tokens BioBERT sees: ['[CLS]', 'the', 'patient', 'presents', 'with', 'short', '##ness', 'of', 'breath', ',', 'w', '##hee', '##zing', ',', 'weakness', ',', 'fever', ',', 'chill', '##s', ',', 'and', 'co', '##ry', '##za', '.', '[SEP]']


In [ ]:
#Dataset Class building the pytorch dataset for the biobert

In [14]:

import torch
from torch.utils.data import Dataset

# Fixed label order — derived from Stage 2 verification
DISEASE_NAMES = [
    "Atelectasis",        # 0
    "Emphysema",          # 1
    "Hiatal Hernia",      # 2
    "Pleural Effusion",   # 3
    "Pneumonia",          # 4
    "Pneumothorax",       # 5
    "Pulmonary Congestion", # 6
    "Pulmonary Fibrosis", # 7
]
NUM_CLASSES = len(DISEASE_NAMES)  # 8


class ChestDiseaseDataset(Dataset):
    """
    Wraps a CSV DataFrame into a PyTorch Dataset for BioBERT.

    Each __getitem__ call:
      1. Grabs one text + label pair
      2. Tokenizes the text using the BioBERT tokenizer
      3. Returns input_ids, attention_mask, and label as tensors

    The DataLoader (Stage 4) will call this repeatedly to build batches.
    """

    def __init__(self, dataframe, tokenizer, max_length=128):
        """
        Args:
            dataframe  : pandas DataFrame with columns 'text' and 'label'
            tokenizer  : loaded BioBERT tokenizer
            max_length : token sequence length (128 is sufficient for symptom sentences)
        """
        # Convert to lists once at init — faster than indexing DataFrame repeatedly
        self.texts      = dataframe["text"].astype(str).tolist()
        self.labels     = dataframe["label"].astype(int).tolist()
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        """Returns total number of samples in this split."""
        return len(self.texts)

    def __getitem__(self, idx):
        """
        Called once per sample during training.
        Returns a dict of tensors that the DataLoader will stack into a batch.
        """
        text  = self.texts[idx]
        label = self.labels[idx]

        # Tokenize: text → input_ids + attention_mask
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",   # pad short sequences to max_length
            truncation=True,        # cut sequences longer than max_length
            return_tensors="pt"     # return PyTorch tensors directly
        )

        return {
            # squeeze(0): tokenizer adds a batch dim we don't need here
            "input_ids":      encoding["input_ids"].squeeze(0),       # shape: (128,)
            "attention_mask": encoding["attention_mask"].squeeze(0),  # shape: (128,)
            "label":          torch.tensor(label, dtype=torch.long)   # scalar
        }

In [ ]:
#test teh dataset class by building the dataset objeyctes

In [15]:

# Build all three dataset objects
train_dataset = ChestDiseaseDataset(train_df, tokenizer, max_length=MAX_LENGTH)
val_dataset   = ChestDiseaseDataset(val_df,   tokenizer, max_length=MAX_LENGTH)
test_dataset  = ChestDiseaseDataset(test_df,  tokenizer, max_length=MAX_LENGTH)

print("Dataset sizes:")
print(f"  Train:      {len(train_dataset)}")
print(f"  Validation: {len(val_dataset)}")
print(f"  Test:       {len(test_dataset)}")

# Pull one sample and verify tensor shapes
sample = train_dataset[0]
print("\nSingle sample tensor shapes:")
print(f"  input_ids:      {sample['input_ids'].shape}   dtype: {sample['input_ids'].dtype}")
print(f"  attention_mask: {sample['attention_mask'].shape}   dtype: {sample['attention_mask'].dtype}")
print(f"  label:          {sample['label']}   dtype: {sample['label'].dtype}")

print("\nSmoke test passed." if sample["input_ids"].shape[0] == MAX_LENGTH else "ERROR: shape mismatch.")

Dataset sizes:
  Train:      2636
  Validation: 565
  Test:       565

Single sample tensor shapes:
  input_ids:      torch.Size([128])   dtype: torch.int64
  attention_mask: torch.Size([128])   dtype: torch.int64
  label:          4   dtype: torch.int64

Smoke test passed.


In [ ]:
# Stage 4 - DATAloader Wraps each Dataset object into a DataLoader, which feeds data to the model in batches during training.

In [ ]:
#Build DataLoaders

In [16]:
from torch.utils.data import DataLoader

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,          # shuffle every epoch so batches differ each time
    num_workers=2,         # parallel workers to load data while GPU trains
    pin_memory=True        # speeds up CPU→GPU tensor transfer
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("DataLoaders created.")
print(f"  Batch size:        {BATCH_SIZE}")
print(f"  Train batches:     {len(train_loader)}")
print(f"  Validation batches:{len(val_loader)}")
print(f"  Test batches:      {len(test_loader)}")

DataLoaders created.
  Batch size:        16
  Train batches:     165
  Validation batches:36
  Test batches:      36


In [ ]:
#test one batch

In [18]:
batch = next(iter(train_loader))

print("One training batch:")
print(f"  input_ids shape:      {batch['input_ids'].shape}")
print(f"  attention_mask shape: {batch['attention_mask'].shape}")
print(f"  labels shape:         {batch['label'].shape}")
print(f"  labels in this batch: {batch['label'].tolist()}")
print()

# Verify label values are all in range [0, 7]
labels = batch["label"]
assert labels.min() >= 0 and labels.max() <= 7, "Label out of range [0,7]"
print("Label range check passed: all labels in [0, 7]")

# Verify tensor dtype
assert batch["input_ids"].dtype == torch.long,  "input_ids dtype error"
assert batch["label"].dtype     == torch.long,  "label dtype error"
print("Dtype check passed.")

print("\nStage 4 complete. DataLoaders are ready.")

One training batch:
  input_ids shape:      torch.Size([16, 128])
  attention_mask shape: torch.Size([16, 128])
  labels shape:         torch.Size([16])
  labels in this batch: [4, 4, 5, 4, 5, 3, 2, 3, 3, 6, 4, 6, 2, 3, 3, 4]

Label range check passed: all labels in [0, 7]
Dtype check passed.

Stage 4 complete. DataLoaders are ready.


In [ ]:
#

In [ ]:
#

In [ ]:
#

In [ ]:
#---------Stage5 Model Bio bert loading

In [ ]:
#biobert classifier defintion

In [19]:
import torch.nn as nn
from transformers import BertModel

class BioBERTClassifier(nn.Module):
    """
    BioBERT fine-tuned for 8-class chest disease classification.

    Architecture:
        BioBERT backbone  →  [CLS] vector (768-dim)
                          →  Dropout(0.3)
                          →  Linear(768 → 8)
                          →  logits  (softmax applied externally)

    Input:
        input_ids      : (batch_size, 128)  token IDs
        attention_mask : (batch_size, 128)  1=real token, 0=padding

    Output:
        logits         : (batch_size, 8)    raw scores, one per disease class
    """

    def __init__(self, model_name, num_classes=8, dropout_rate=0.3):
        super(BioBERTClassifier, self).__init__()

        # Pre-trained BioBERT backbone — all 12 transformer layers
        self.bert = BertModel.from_pretrained(model_name)

        # Dropout for regularization
        # 0.3 is appropriate for small fine-tuning datasets
        self.dropout = nn.Dropout(dropout_rate)

        # Classification head: maps 768-dim CLS vector → 8 class scores
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        """
        Forward pass.

        We use self.bert.config.hidden_size (768) instead of hardcoding 768
        so this works even if the backbone is swapped to a larger BERT variant.
        """
        # Pass tokenized input through all 12 BioBERT layers
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # outputs.last_hidden_state : (batch_size, 128, 768)
        # We take position 0 → the [CLS] token for every sample in the batch
        cls_vector = outputs.last_hidden_state[:, 0, :]  # (batch_size, 768)

        # Apply dropout
        cls_vector = self.dropout(cls_vector)

        # Project to 8 class scores
        logits = self.classifier(cls_vector)  # (batch_size, 8)

        return logits

In [ ]:
# create instance of model and verify

In [20]:
print(f"Loading BioBERT backbone: {MODEL_NAME}")
print("This downloads ~440 MB on first run. Subsequent runs use the cache.\n")

model = BioBERTClassifier(
    model_name=MODEL_NAME,
    num_classes=NUM_CLASSES,   # 8, defined in Stage 3
    dropout_rate=0.3
)

# Move to GPU
model = model.to(device)

# ── Parameter count ──
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
classifier_params = sum(p.numel() for p in model.classifier.parameters())

print("=" * 50)
print("MODEL SUMMARY")
print("=" * 50)
print(f"  Total parameters:      {total_params:>12,}")
print(f"  Trainable parameters:  {trainable_params:>12,}")
print(f"  Classifier head only:  {classifier_params:>12,}   (Linear 768→8)")
print(f"  Device:                {next(model.parameters()).device}")
print(f"  Hidden size:           {model.bert.config.hidden_size}")
print(f"  BERT layers:           {model.bert.config.num_hidden_layers}")
print(f"  Output classes:        {NUM_CLASSES}")

Loading BioBERT backbone: dmis-lab/biobert-base-cased-v1.1
This downloads ~440 MB on first run. Subsequent runs use the cache.



config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  436MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dmis-lab/biobert-base-cased-v1.1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MODEL SUMMARY
  Total parameters:       108,316,424
  Trainable parameters:   108,316,424
  Classifier head only:         6,152   (Linear 768→8)
  Device:                cuda:0
  Hidden size:           768
  BERT layers:           12
  Output classes:        8


In [ ]:
# forward pass test the model and use the previous batch we created from DataLoader stage

In [21]:

model.eval()  # disable dropout for this test

# Reuse the batch we already have from Stage 4
test_input_ids  = batch["input_ids"].to(device)
test_attn_mask  = batch["attention_mask"].to(device)
test_labels     = batch["label"].to(device)

with torch.no_grad():
    logits = model(test_input_ids, test_attn_mask)

print("Forward pass output:")
print(f"  Input shape:   {test_input_ids.shape}")
print(f"  Output shape:  {logits.shape}   ← should be (16, 8)")
print(f"  Output dtype:  {logits.dtype}")
print()

# Convert logits → probabilities for one sample to verify they sum to 1
probs = torch.softmax(logits[0], dim=0)
print("Probabilities for sample 0 (untrained — random-looking is expected):")
for i, (name, prob) in enumerate(zip(DISEASE_NAMES, probs)):
    bar = "█" * int(prob * 40)
    print(f"  [{i}] {name:<25} {prob:.4f}  {bar}")
print(f"\n  Sum of probabilities: {probs.sum().item():.6f}   ← must be 1.000000")

assert logits.shape == (BATCH_SIZE, NUM_CLASSES), "Output shape mismatch"
assert abs(probs.sum().item() - 1.0) < 1e-5, "Probabilities do not sum to 1"
print("\nForward pass smoke test passed.")

model.train()  # restore training mode

Forward pass output:
  Input shape:   torch.Size([16, 128])
  Output shape:  torch.Size([16, 8])   ← should be (16, 8)
  Output dtype:  torch.float32

Probabilities for sample 0 (untrained — random-looking is expected):
  [0] Atelectasis               0.1362  █████
  [1] Emphysema                 0.0995  ███
  [2] Hiatal Hernia             0.0945  ███
  [3] Pleural Effusion          0.0846  ███
  [4] Pneumonia                 0.1639  ██████
  [5] Pneumothorax              0.1169  ████
  [6] Pulmonary Congestion      0.1399  █████
  [7] Pulmonary Fibrosis        0.1644  ██████

  Sum of probabilities: 1.000000   ← must be 1.000000

Forward pass smoke test passed.


BioBERTClassifier(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise

In [ ]:
# up untill now we can see the dataset is properly loading , tokenizer is working , biobert is laoding properluy and it is also working
#on the fwd pass we give it symptom it gives 8 probailites(although not fine tuned yet because this has now random weights for classification)


In [ ]:
#stage 6 claass weights so since here problem is that we have minority classes so we need to define weights that
# in loss defintion it assgins loss accroding to weigths of each class assign higher weiughts to minor class to balance

In [ ]:
#so 1st fine the weights

In [22]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# Extract all training labels as a numpy array
train_labels = train_df["label"].values

# Compute balanced class weights
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),  # [0, 1, 2, 3, 4, 5, 6, 7]
    y=train_labels
)

print("=" * 52)
print("CLASS WEIGHTS")
print("=" * 52)
print(f"  {'Label':<6} {'Disease':<25} {'Count':>6} {'Weight':>8}")
print(f"  {'-'*5} {'-'*24} {'-'*6} {'-'*8}")

label_counts = train_df["label"].value_counts().sort_index()

for i, (name, weight) in enumerate(zip(DISEASE_NAMES, class_weights)):
    count = label_counts[i]
    bar   = "█" * int(weight * 3)
    print(f"  [{i}]   {name:<25} {count:>6} {weight:>8.4f}  {bar}")

print()
print("Interpretation:")
print("  Higher weight = model penalized more for getting this class wrong.")
print("  Lower  weight = class is common enough to learn naturally.")

# Convert to tensor on GPU — loss function needs this format
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
print(f"\nWeights tensor device: {class_weights_tensor.device}")
print(f"Weights tensor: {class_weights_tensor}")

CLASS WEIGHTS
  Label  Disease                    Count   Weight
  ----- ------------------------ ------ --------
  [0]   Atelectasis                   69   4.7754  ██████████████
  [1]   Emphysema                     22  14.9773  ████████████████████████████████████████████
  [2]   Hiatal Hernia                634   0.5197  █
  [3]   Pleural Effusion             425   0.7753  ██
  [4]   Pneumonia                    848   0.3886  █
  [5]   Pneumothorax                 207   1.5918  ████
  [6]   Pulmonary Congestion         348   0.9468  ██
  [7]   Pulmonary Fibrosis            83   3.9699  ███████████

Interpretation:
  Higher weight = model penalized more for getting this class wrong.
  Lower  weight = class is common enough to learn naturally.

Weights tensor device: cuda:0
Weights tensor: tensor([ 4.7754, 14.9773,  0.5197,  0.7753,  0.3886,  1.5918,  0.9468,  3.9699],
       device='cuda:0')


In [ ]:
#Define the loss function WeightedCrossEntropy that takes these weights and computes loss acorddingly so no need of softmax
#separately because it handles internally

In [23]:
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

print("Loss function: CrossEntropyLoss with class weights")
print(f"Weight tensor shape: {class_weights_tensor.shape}")

# ── Quick sanity check ──
# Pass the same test batch through the loss function
# with the untrained model output we already have
model.eval()
with torch.no_grad():
    test_logits = model(
        batch["input_ids"].to(device),
        batch["attention_mask"].to(device)
    )
    test_loss = criterion(test_logits, batch["label"].to(device))

print(f"\nLoss on one untrained batch: {test_loss.item():.4f}")
print("(Expected: around 2.0–2.5 for random predictions on 8 classes)")
print("\nStage 6 complete.")

model.train()

Loss function: CrossEntropyLoss with class weights
Weight tensor shape: torch.Size([8])

Loss on one untrained batch: 2.1953
(Expected: around 2.0–2.5 for random predictions on 8 classes)

Stage 6 complete.


BioBERTClassifier(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise

In [ ]:
#Stage7 Optimizer and Scheduler

In [ ]:
# so now here after loss has been computed we will have to update the weights on each fwd pass that will be handled
#by the optimizer and scheduler will decide how learning rate changes across trainign (it wont stay static throughout)


In [24]:
# ============================================================
# Stage 7 | Cell 18 — Optimizer and Scheduler
# ============================================================
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW

# ── Training hyperparameters ──
NUM_EPOCHS    = 5
LEARNING_RATE = 2e-5
WARMUP_RATIO  = 0.1   # first 10% of steps used for warmup

# ── Optimizer ──
# weight_decay regularizes large weights to prevent overfitting
# We exclude bias and LayerNorm parameters from decay — standard practice for BERT
no_decay = ["bias", "LayerNorm.weight"]

optimizer_grouped_parameters = [
    {
        # All parameters that SHOULD have weight decay
        "params": [
            p for n, p in model.named_parameters()
            if not any(nd in n for nd in no_decay)
        ],
        "weight_decay": 0.01,
    },
    {
        # Bias and LayerNorm — NO weight decay
        "params": [
            p for n, p in model.named_parameters()
            if any(nd in n for nd in no_decay)
        ],
        "weight_decay": 0.0,
    },
]

optimizer = AdamW(optimizer_grouped_parameters, lr=LEARNING_RATE)

# ── Scheduler ──
total_steps  = len(train_loader) * NUM_EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print("=" * 45)
print("OPTIMIZER AND SCHEDULER")
print("=" * 45)
print(f"  Optimizer:        AdamW")
print(f"  Learning rate:    {LEARNING_RATE}")
print(f"  Weight decay:     0.01 (excluding bias and LayerNorm)")
print(f"  Epochs:           {NUM_EPOCHS}")
print(f"  Batches/epoch:    {len(train_loader)}")
print(f"  Total steps:      {total_steps}")
print(f"  Warmup steps:     {warmup_steps}  ({WARMUP_RATIO*100:.0f}% of total)")
print(f"\nStage 7 complete. Ready to train.")

OPTIMIZER AND SCHEDULER
  Optimizer:        AdamW
  Learning rate:    2e-05
  Weight decay:     0.01 (excluding bias and LayerNorm)
  Epochs:           5
  Batches/epoch:    165
  Total steps:      825
  Warmup steps:     82  (10% of total)

Stage 7 complete. Ready to train.


In [ ]:
#stage8   Training Pipelien

In [ ]:
#Train single Epoch

In [25]:
# ============================================================
# Stage 8 | Cell 19 — Train One Epoch Function
# ============================================================

def train_one_epoch(model, loader, optimizer, scheduler, criterion, device):
    """
    Runs one full pass over the training data.

    Returns:
        avg_loss : average loss across all batches
        accuracy : overall training accuracy for this epoch
    """
    model.train()   # activates dropout layers

    total_loss = 0.0
    correct    = 0
    total      = 0

    for batch_idx, batch in enumerate(loader):

        # Move everything to GPU
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)

        # ── Forward pass ──
        optimizer.zero_grad()                      # clear gradients from last step
        logits = model(input_ids, attention_mask)  # shape: (batch_size, 8)
        loss   = criterion(logits, labels)         # weighted cross entropy loss

        # ── Backward pass ──
        loss.backward()                            # compute gradients

        # Clip gradients — standard for transformer fine-tuning
        # prevents any single gradient from becoming too large
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()   # update weights
        scheduler.step()   # update learning rate

        # ── Track metrics ──
        total_loss += loss.item()
        preds       = torch.argmax(logits, dim=1)       # predicted class per sample
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

        # Progress print every 50 batches so you know it is running
        if (batch_idx + 1) % 50 == 0:
            print(f"    Batch {batch_idx+1:>3}/{len(loader)} "
                  f"| Batch Loss: {loss.item():.4f} "
                  f"| LR: {scheduler.get_last_lr()[0]:.2e}")

    avg_loss = total_loss / len(loader)
    accuracy = correct / total

    return avg_loss, accuracy

In [26]:
# ============================================================
# Stage 8 | Cell 20 — Evaluate Function
# ============================================================
from sklearn.metrics import f1_score, classification_report

def evaluate(model, loader, criterion, device):
    """
    Runs one full pass over validation or test data.
    No weight updates — purely measuring performance.

    Returns:
        avg_loss    : average weighted cross entropy loss
        accuracy    : overall accuracy
        macro_f1    : macro F1 score across all 8 classes
                      (treats every class equally regardless of size)
        all_preds   : list of predicted labels (for detailed report later)
        all_labels  : list of true labels (for detailed report later)
    """
    model.eval()   # deactivates dropout — deterministic output

    total_loss = 0.0
    correct    = 0
    total      = 0
    all_preds  = []
    all_labels = []

    with torch.no_grad():   # no gradients needed during evaluation
        for batch in loader:

            # Move to GPU
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["label"].to(device)

            # Forward pass only — no backward
            logits = model(input_ids, attention_mask)
            loss   = criterion(logits, labels)

            # Track loss
            total_loss += loss.item()

            # Track predictions
            preds    = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

            # Collect all predictions and labels for F1 computation
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    accuracy = correct / total

    # Macro F1 — computes F1 per class then averages
    # This is what we use for checkpoint saving
    # zero_division=0 handles classes with no predictions gracefully
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    return avg_loss, accuracy, macro_f1, all_preds, all_labels

In [ ]:
#Now Full trainging script that actualkly start training each epoch by epoch

In [ ]:
# ============================================================
# Stage 8 | Cell 21 — Full Training Loop
# ============================================================
import json
import time

# ── Paths for saving ──
BEST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, "best_model.pt")
LAST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, "last_model.pt")
HISTORY_PATH    = os.path.join(LOG_DIR, "training_history.json")

# ── Training history — saved to logs after training ──
history = {
    "train_loss"      : [],
    "train_accuracy"  : [],
    "val_loss"        : [],
    "val_accuracy"    : [],
    "val_macro_f1"    : [],
    "best_epoch"      : None,
    "best_val_macro_f1" : None
}

# ── Tracking best model ──
best_val_macro_f1 = -1.0   # any real F1 will be higher than this

print("=" * 60)
print("STARTING TRAINING")
print("=" * 60)
print(f"  Epochs:           {NUM_EPOCHS}")
print(f"  Train batches:    {len(train_loader)} per epoch")
print(f"  Val batches:      {len(val_loader)} per epoch")
print(f"  Best model path:  {BEST_MODEL_PATH}")
print(f"  Last model path:  {LAST_MODEL_PATH}")
print(f"  History path:     {HISTORY_PATH}")
print("=" * 60)

# ── Main training loop ──
for epoch in range(1, NUM_EPOCHS + 1):

    epoch_start = time.time()

    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")
    print("-" * 60)

    # ── Train ──
    print("  Training...")
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, scheduler, criterion, device
    )

    # ── Validate ──
    print("  Evaluating on validation set...")
    val_loss, val_acc, val_macro_f1, val_preds, val_labels = evaluate(
        model, val_loader, criterion, device
    )

    epoch_time = time.time() - epoch_start

    # ── Print epoch summary ──
    print(f"\n  Epoch {epoch} Summary:")
    print(f"  {'Metric':<25} {'Train':>10} {'Validation':>12}")
    print(f"  {'-'*25} {'-'*10} {'-'*12}")
    print(f"  {'Loss':<25} {train_loss:>10.4f} {val_loss:>12.4f}")
    print(f"  {'Accuracy':<25} {train_acc*100:>9.2f}% {val_acc*100:>11.2f}%")
    print(f"  {'Macro F1':<25} {'':>10} {val_macro_f1:>12.4f}")
    print(f"  {'Time':<25} {epoch_time:>9.1f}s")

    # ── Save history ──
    history["train_loss"].append(round(train_loss, 6))
    history["train_accuracy"].append(round(train_acc, 6))
    history["val_loss"].append(round(val_loss, 6))
    history["val_accuracy"].append(round(val_acc, 6))
    history["val_macro_f1"].append(round(val_macro_f1, 6))

    # ── Save best model if validation Macro F1 improved ──
    if val_macro_f1 > best_val_macro_f1:
        best_val_macro_f1          = val_macro_f1
        history["best_epoch"]      = epoch
        history["best_val_macro_f1"] = round(best_val_macro_f1, 6)

        torch.save({
            "epoch"          : epoch,
            "model_state_dict"   : model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_macro_f1"   : val_macro_f1,
            "val_loss"       : val_loss,
            "disease_names"  : DISEASE_NAMES,
            "num_classes"    : NUM_CLASSES,
            "model_name"     : MODEL_NAME,
        }, BEST_MODEL_PATH)

        print(f"\n  ✓ Best model saved — Val Macro F1 improved to {val_macro_f1:.4f}")
    else:
        print(f"\n  No improvement. Best so far: {best_val_macro_f1:.4f} "
              f"(Epoch {history['best_epoch']})")

    # ── Always save last model ──
    torch.save({
        "epoch"              : epoch,
        "model_state_dict"   : model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "val_macro_f1"       : val_macro_f1,
        "val_loss"           : val_loss,
        "disease_names"      : DISEASE_NAMES,
        "num_classes"        : NUM_CLASSES,
        "model_name"         : MODEL_NAME,
    }, LAST_MODEL_PATH)

# ── Save training history to logs ──
with open(HISTORY_PATH, "w") as f:
    json.dump(history, f, indent=2)

# ── Final summary ──
print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"  Best epoch:          {history['best_epoch']}")
print(f"  Best Val Macro F1:   {history['best_val_macro_f1']:.4f}")
print(f"  Best model saved to: {BEST_MODEL_PATH}")
print(f"  Last model saved to: {LAST_MODEL_PATH}")
print(f"  History saved to:    {HISTORY_PATH}")
print("=" * 60)

STARTING TRAINING
  Epochs:           5
  Train batches:    165 per epoch
  Val batches:      36 per epoch
  Best model path:  /content/drive/MyDrive/BioBERT_Project/checkpoints/best_model.pt
  Last model path:  /content/drive/MyDrive/BioBERT_Project/checkpoints/last_model.pt
  History path:     /content/drive/MyDrive/BioBERT_Project/logs/training_history.json

Epoch 1/5
------------------------------------------------------------
  Training...
    Batch  50/165 | Batch Loss: 1.6089 | LR: 1.22e-05
    Batch 100/165 | Batch Loss: 0.2226 | LR: 1.95e-05
    Batch 150/165 | Batch Loss: 0.0184 | LR: 1.82e-05
  Evaluating on validation set...

  Epoch 1 Summary:
  Metric                         Train   Validation
  ------------------------- ---------- ------------
  Loss                          1.0247       0.1674
  Accuracy                      70.30%       96.81%
  Macro F1                                   0.9248
  Time                           76.0s

  ✓ Best model saved — Val Macro F1

In [ ]:
# already covered and fine tuned
"""
Epoch   Train Loss   Val Loss   Val Accuracy   Val Macro F1
─────   ──────────   ────────   ────────────   ────────────
  1       1.0247      0.1674       96.81%          0.9248
  2       0.0883      0.1605       97.35%          0.9323   ← saved
  3       0.0692      0.1510       97.17%          0.9315   ← no improvement
  4       0.0474      0.1637       97.88%          0.9579   ← saved (best)
  5       0.0375      0.1742       97.52%          0.9546   ← no improvement

  """


'\nEpoch   Train Loss   Val Loss   Val Accuracy   Val Macro F1\n─────   ──────────   ────────   ────────────   ────────────\n  1       1.0247      0.1674       96.81%          0.9248\n  2       0.0883      0.1605       97.35%          0.9323   ← saved\n  3       0.0692      0.1510       97.17%          0.9315   ← no improvement\n  4       0.0474      0.1637       97.88%          0.9579   ← saved (best)\n  5       0.0375      0.1742       97.52%          0.9546   ← no improvement\n\n  '

In [ ]:
#

In [ ]:
#

In [ ]:
#

In [ ]:
# evalaute separately on common and uncommon cases

In [1]:
print("Hello")

Hello


In [2]:
#

In [3]:
#

In [4]:
#

In [39]:
# ============================================================
# Load Best Checkpoint from Drive
# ============================================================

BEST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, "best_model.pt")

print(f"Loading checkpoint from: {BEST_MODEL_PATH}")

checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

print(f"Checkpoint loaded successfully.")
print(f"  Saved from epoch:   {checkpoint['epoch']}")
print(f"  Val Macro F1:       {checkpoint['val_macro_f1']:.4f}")
print(f"  Val Loss:           {checkpoint['val_loss']:.4f}")

Loading checkpoint from: /content/drive/MyDrive/BioBERT_Project/checkpoints/best_model.pt
Checkpoint loaded successfully.
  Saved from epoch:   4
  Val Macro F1:       0.9579
  Val Loss:           0.1637


In [42]:
# ============================================================
# Full Test Set Evaluation — Complete Report
# ============================================================
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

print("Running evaluation on full test set...")
test_loss, test_acc, test_macro_f1, test_preds, test_labels = evaluate(
    model, test_loader, criterion, device
)

# ── Overall metrics ──
print("\n" + "=" * 55)
print("TEST SET RESULTS — OVERALL")
print("=" * 55)
print(f"  Test Loss:        {test_loss:.4f}")
print(f"  Test Accuracy:    {test_acc*100:.2f}%")
print(f"  Test Macro F1:    {test_macro_f1:.4f}")

# ── Per-class classification report ──
print("\n" + "=" * 55)
print("PER-CLASS CLASSIFICATION REPORT")
print("=" * 55)
print(classification_report(
    test_labels,
    test_preds,
    target_names=DISEASE_NAMES,
    digits=4,
    zero_division=0
))

# ── Confusion matrix ──
print("=" * 55)
print("CONFUSION MATRIX")
print("(Rows = Actual class, Columns = Predicted class)")
print("=" * 55)

cm = confusion_matrix(test_labels, test_preds)

# Print header row with short disease names
short_names = ["Atel", "Emph", "HHer", "PlEf", "Pneu", "Pntx", "PCon", "PFib"]
header = f"  {'Actual → Pred':<22}"
for name in short_names:
    header += f"{name:>7}"
print(header)
print("  " + "-" * 78)

# Print each row
for i, row in enumerate(cm):
    row_str = f"  {DISEASE_NAMES[i][:22]:<22}"
    for j, val in enumerate(row):
        if i == j:
            row_str += f"{'✓'+str(val):>7}"   # correct predictions marked
        else:
            row_str += f"{val:>7}"
    print(row_str)

print()
print("  ✓ = correct predictions (diagonal)")
print("  Any other number = misclassification")

# ── Per-class accuracy summary ──
print("\n" + "=" * 55)
print("PER-CLASS ACCURACY SUMMARY")
print("=" * 55)
print(f"  {'Disease':<25} {'Correct':>8} {'Total':>7} {'Accuracy':>10}")
print(f"  {'-'*25} {'-'*8} {'-'*7} {'-'*10}")
for i in range(NUM_CLASSES):
    correct_i = cm[i][i]
    total_i   = cm[i].sum()
    acc_i     = correct_i / total_i * 100 if total_i > 0 else 0
    print(f"  {DISEASE_NAMES[i]:<25} {correct_i:>8} {total_i:>7} {acc_i:>9.2f}%")

Running evaluation on full test set...

TEST SET RESULTS — OVERALL
  Test Loss:        0.0881
  Test Accuracy:    98.05%
  Test Macro F1:    0.9640

PER-CLASS CLASSIFICATION REPORT
                      precision    recall  f1-score   support

         Atelectasis     1.0000    1.0000    1.0000        15
           Emphysema     0.8333    1.0000    0.9091         5
       Hiatal Hernia     1.0000    1.0000    1.0000       136
    Pleural Effusion     0.9579    1.0000    0.9785        91
           Pneumonia     0.9944    0.9670    0.9805       182
        Pneumothorax     1.0000    0.9773    0.9885        44
Pulmonary Congestion     0.9730    0.9600    0.9664        75
  Pulmonary Fibrosis     0.8421    0.9412    0.8889        17

            accuracy                         0.9805       565
           macro avg     0.9501    0.9807    0.9640       565
        weighted avg     0.9816    0.9805    0.9808       565

CONFUSION MATRIX
(Rows = Actual class, Columns = Predicted class)
  Actu

In [41]:
# ============================================================
# Full Overlap Analysis
# ============================================================
import numpy as np

# ── Step 1: Identify clean vs overlap ──
train_texts    = set(train_df["text"].str.strip().str.lower())
test_df["text_lower"] = test_df["text"].str.strip().str.lower()
test_df["in_train"]   = test_df["text_lower"].isin(train_texts)

clean_test   = test_df[test_df["in_train"] == False].copy()
overlap_test = test_df[test_df["in_train"] == True].copy()

print("=" * 55)
print("TEST SET SPLIT")
print("=" * 55)
print(f"  Total test samples:        {len(test_df)}")
print(f"  Clean  (never seen):       {len(clean_test)}")
print(f"  Overlap (seen in train):   {len(overlap_test)}")

# ── Step 2: Build loaders ──
clean_dataset   = ChestDiseaseDataset(clean_test,   tokenizer, max_length=MAX_LENGTH)
overlap_dataset = ChestDiseaseDataset(overlap_test, tokenizer, max_length=MAX_LENGTH)

clean_loader   = DataLoader(clean_dataset,   batch_size=BATCH_SIZE, shuffle=False)
overlap_loader = DataLoader(overlap_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ── Step 3: Evaluate both ──
print("\nEvaluating clean samples...")
clean_loss, clean_acc, clean_f1, clean_preds, clean_labels = evaluate(
    model, clean_loader, criterion, device
)

print("Evaluating overlap samples...")
over_loss, over_acc, over_f1, over_preds, over_labels = evaluate(
    model, overlap_loader, criterion, device
)

# ── Step 4: Comparison table ──
print("\n" + "=" * 58)
print("RESULTS COMPARISON")
print("=" * 58)
print(f"  {'Metric':<20} {'Full Test':>10} {'Clean Only':>12} {'Overlap Only':>14}")
print(f"  {'-'*20} {'-'*10} {'-'*12} {'-'*14}")
print(f"  {'Accuracy':<20} {test_acc*100:>9.2f}% {clean_acc*100:>11.2f}% {over_acc*100:>13.2f}%")
print(f"  {'Macro F1':<20} {test_macro_f1:>10.4f} {clean_f1:>12.4f} {over_f1:>14.4f}")
print(f"  {'Loss':<20} {test_loss:>10.4f} {clean_loss:>12.4f} {over_loss:>14.4f}")
print(f"  {'Samples':<20} {len(test_df):>10} {len(clean_test):>12} {len(overlap_test):>14}")

# ── Step 5: Per-class report clean only ──
print("\n" + "=" * 55)
print("PER-CLASS REPORT — CLEAN TEST ONLY")
print("=" * 55)

present_labels = sorted(list(set(clean_labels)))
present_names  = [DISEASE_NAMES[i] for i in present_labels]
missing_labels = [i for i in range(NUM_CLASSES) if i not in present_labels]

if missing_labels:
    print(f"  Classes missing from clean subset:")
    for i in missing_labels:
        print(f"    - {DISEASE_NAMES[i]}")
    print()

print(classification_report(
    clean_labels,
    clean_preds,
    labels=present_labels,
    target_names=present_names,
    digits=4,
    zero_division=0
))

# ── Step 6: Per-class report overlap only ──
print("=" * 55)
print("PER-CLASS REPORT — OVERLAP TEST ONLY")
print("=" * 55)

present_labels_ov = sorted(list(set(over_labels)))
present_names_ov  = [DISEASE_NAMES[i] for i in present_labels_ov]
missing_labels_ov = [i for i in range(NUM_CLASSES) if i not in present_labels_ov]

if missing_labels_ov:
    print(f"  Classes missing from overlap subset:")
    for i in missing_labels_ov:
        print(f"    - {DISEASE_NAMES[i]}")
    print()

print(classification_report(
    over_labels,
    over_preds,
    labels=present_labels_ov,
    target_names=present_names_ov,
    digits=4,
    zero_division=0
))

# ── Step 7: Class distribution ──
print("=" * 55)
print("CLASS DISTRIBUTION IN EACH SUBSET")
print("=" * 55)
print(f"  {'Disease':<25} {'Clean':>8} {'Overlap':>10}")
print(f"  {'-'*25} {'-'*8} {'-'*10}")

clean_counts   = clean_test["label"].value_counts().sort_index()
overlap_counts = overlap_test["label"].value_counts().sort_index()

for i in range(NUM_CLASSES):
    c = clean_counts.get(i, 0)
    o = overlap_counts.get(i, 0)
    print(f"  {DISEASE_NAMES[i]:<25} {c:>8} {o:>10}")

TEST SET SPLIT
  Total test samples:        565
  Clean  (never seen):       428
  Overlap (seen in train):   137

Evaluating clean samples...
Evaluating overlap samples...

RESULTS COMPARISON
  Metric                Full Test   Clean Only   Overlap Only
  -------------------- ---------- ------------ --------------
  Accuracy                 98.05%       99.30%         94.16%
  Macro F1                 0.9640       0.9813         0.8338
  Loss                     0.0881       0.0387         0.1409
  Samples                     565          428            137

PER-CLASS REPORT — CLEAN TEST ONLY
  Classes missing from clean subset:
    - Emphysema

                      precision    recall  f1-score   support

         Atelectasis     1.0000    1.0000    1.0000         4
       Hiatal Hernia     1.0000    1.0000    1.0000       121
    Pleural Effusion     0.9855    1.0000    0.9927        68
           Pneumonia     1.0000    0.9888    0.9944       178
        Pneumothorax     1.0000   

In [43]:
#


In [44]:
#

In [45]:
#

In [47]:
# ============================================================
# Temperature Scaling — Find Best Temperature
# ============================================================

model.eval()

# Collect all logits from validation set first
all_logits = []
all_labels_val = []

with torch.no_grad():
    for batch in val_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"]

        logits = model(input_ids, attention_mask)
        all_logits.append(logits.cpu())
        all_labels_val.extend(labels.numpy())

all_logits = torch.cat(all_logits, dim=0)  # (565, 8)

# Try different temperature values
temperatures = [1, 2, 3, 4, 5, 7, 10, 15, 20]

print("=" * 65)
print("TEMPERATURE SCALING ANALYSIS")
print("=" * 65)
print(f"  {'Temp':>6}  {'Avg Entropy':>12}  {'Avg Max Prob':>13}  {'Val Accuracy':>13}")
print(f"  {'-'*6}  {'-'*12}  {'-'*13}  {'-'*13}")

for T in temperatures:
    # Apply temperature
    scaled_probs = torch.softmax(all_logits / T, dim=1).numpy()

    # Compute average entropy
    entropy_per_sample = -np.sum(
        scaled_probs * np.log(scaled_probs + 1e-10), axis=1
    )
    avg_entropy = entropy_per_sample.mean()

    # Compute average max probability
    avg_max_prob = scaled_probs.max(axis=1).mean()

    # Compute accuracy (temperature does not change predicted class)
    preds_T = scaled_probs.argmax(axis=1)
    acc_T   = (preds_T == np.array(all_labels_val)).mean() * 100

    print(f"  {T:>6}  {avg_entropy:>12.4f}  {avg_max_prob:>13.4f}  {acc_T:>12.2f}%")

print()
print(f"  Target entropy range for fusion: 0.5 — 1.5")
print(f"  Max entropy possible (uniform):  {np.log(NUM_CLASSES):.4f}")

TEMPERATURE SCALING ANALYSIS
    Temp   Avg Entropy   Avg Max Prob   Val Accuracy
  ------  ------------  -------------  -------------
       1        0.0205         0.9946         97.88%
       2        0.3620         0.9313         97.88%
       3        1.0016         0.7608         97.88%
       4        1.4490         0.5973         97.88%
       5        1.6939         0.4825         97.88%
       7        1.9050         0.3534         97.88%
      10        2.0046         0.2677         97.88%
      15        2.0501         0.2109         97.88%
      20        2.0640         0.1860         97.88%

  Target entropy range for fusion: 0.5 — 1.5
  Max entropy possible (uniform):  2.0794
